# Folio-OCR on free GPUs: Phase 2 (general SFT), Phase 3 (RL), Phase 4 (merge, quantise)

Runs on **Kaggle** (*Accelerator* = **GPU T4 x2**, *Internet* = **On**). It continues from Phase 1, so add these as Kaggle datasets:

- the code (`Folio-OCR.zip`),
- your Phase 1 adapter (`runs/resume-t4/best` from the first notebook's output),
- the pages you rendered on your own computer (`data/`: `resumes`, `render_old`, `render_layout`, `dev_old`, `dev_multicol`; the commands are in `docs/PHASES.md`).

Each section ends at a gate of the plan (G2, G3, G4); `docs/PHASES.md` has the pass rules. Long steps resume:
converters skip pages they already wrote, benchmarks skip finished pages, training continues from its last checkpoint.
Kaggle gives about 30 GPU-hours a week, so expect this notebook to span several sessions.

In [ ]:
import os, glob, shutil, subprocess, sys, zipfile, json
ON_KAGGLE = os.path.exists("/kaggle")
WORK = "/kaggle/working" if ON_KAGGLE else "/content"
CODE, RUNS = f"{WORK}/Folio-OCR", f"{WORK}/runs"
N_GPUS = int(subprocess.run("nvidia-smi -L | wc -l", shell=True, capture_output=True, text=True).stdout.strip() or 0)
print("GPUs:", N_GPUS)

# ---- set these to your Kaggle inputs ----
PHASE1_ADAPTER = "/kaggle/input/folio-resume-adapter/best"   # Phase 1 output (adapter folder)
RENDERED = "/kaggle/input/folio-data/data"                   # rendered pages (resumes, render_old, ...)

In [ ]:
if not os.path.exists(f"{CODE}/folioocr"):
    found = glob.glob("/kaggle/input/**/folioocr/__init__.py", recursive=True)
    zips = glob.glob("/kaggle/input/**/Folio-OCR*.zip", recursive=True)
    if found:
        shutil.copytree(os.path.dirname(os.path.dirname(found[0])), CODE)
    elif zips:
        zipfile.ZipFile(zips[0]).extractall(WORK)
os.chdir(CODE)
!pip install -q "transformers==5.21.0" "peft==0.21.2" "accelerate>=1.10" "bitsandbytes>=0.45" \
    pypdfium2 faker fonttools rapidfuzz fuzzysearch beautifulsoup4 pypdf playwright apted
!pip install -q --no-deps "olmocr==0.4.27"
!python -m pytest -q tests 2>&1 | tail -2
DATA = f"{WORK}/data"
os.makedirs(DATA, exist_ok=True)
for d in glob.glob(f"{RENDERED}/*"):              # rendered sets are read in place (inputs are read-only)
    if not os.path.exists(f"{DATA}/{os.path.basename(d)}"):
        os.symlink(d, f"{DATA}/{os.path.basename(d)}")
!ls -la {DATA}

## Phase 2a. Public data

One tarball per `olmOCR-mix` subset (~1 GB each, a few thousand pages) and all of `olmOCR-synthmix` (1.2 GB).
The historical subsets (`02_loc_transcripts`, `03_national_archives`) are the closest public match to olmOCR-Bench's old scans.

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download("allenai/olmOCR-mix-1025", repo_type="dataset", local_dir=f"{WORK}/olmocr-mix", allow_patterns=[
    "00_documents_train.parquet", "pdf_tarballs/00_documents_train_00000.tar.gz",
    "02_loc_transcripts_train.parquet", "pdf_tarballs/02_loc_transcripts_train_00000.tar.gz",
    "03_national_archives_train.parquet", "pdf_tarballs/03_national_archives_train_00000.tar.gz",
    "02_loc_transcripts_eval.parquet", "pdf_tarballs/02_loc_transcripts_eval_*"])
snapshot_download("allenai/olmOCR-synthmix-1025", repo_type="dataset", local_dir=f"{WORK}/synthmix")
snapshot_download("allenai/olmOCR-bench", repo_type="dataset", local_dir=f"{WORK}/olmOCR-bench")

In [ ]:
M = f"{WORK}/olmocr-mix"
!python scripts/prepare_data.py olmocr-mix --parquet {M}/00_documents_train.parquet --tarballs {M}/pdf_tarballs/00_documents_train_00000.tar.gz --out {DATA}/olmocr_docs --limit 3000
!python scripts/prepare_data.py olmocr-mix --parquet {M}/02_loc_transcripts_train.parquet --tarballs {M}/pdf_tarballs/02_loc_transcripts_train_00000.tar.gz --out {DATA}/olmocr_hist --limit 1500
!python scripts/prepare_data.py olmocr-mix --parquet {M}/03_national_archives_train.parquet --tarballs {M}/pdf_tarballs/03_national_archives_train_00000.tar.gz --out {DATA}/olmocr_hist --limit 1500
!python scripts/prepare_data.py olmocr-mix --parquet {M}/02_loc_transcripts_eval.parquet --tarballs {M}/pdf_tarballs/02_loc_transcripts_eval_* --out {DATA}/dev_loc --dev-frac 1
!python scripts/prepare_data.py synthmix --root {WORK}/synthmix --out-sft {DATA}/synthmix --out-rl {DATA}/rl_synthmix
!python scripts/prepare_data.py preview --labels {DATA}/olmocr_hist/labels.jsonl --n 6

## Phase 2b. Contamination check and data card

Every evaluation page is indexed (image hash, text MinHash, benchmark test strings, file hashes); every training source
is checked against it, and only the `labels.clean.jsonl` files are trained on (the mixture presets point at them).
Our rendered sets live in read-only inputs, so their clean copies are written next to a writable copy of the label file.

In [ ]:
for d in ["render_old", "render_layout"]:                      # writable label files beside read-only images
    src = os.path.realpath(f"{DATA}/{d}")
    if os.path.islink(f"{DATA}/{d}"):
        os.unlink(f"{DATA}/{d}"); os.makedirs(f"{DATA}/{d}")
        for f in os.listdir(src):
            (shutil.copy if f.endswith((".jsonl", ".json")) else os.symlink)(os.path.join(src, f), f"{DATA}/{d}/{f}")
!python scripts/dedup.py index --out {WORK}/eval_index.npz --olmocr {WORK}/olmOCR-bench/bench_data \
    --labels dev_old={DATA}/dev_old/labels.jsonl dev_multicol={DATA}/dev_multicol/labels.jsonl dev_loc={DATA}/dev_loc/labels.jsonl
!python scripts/dedup.py check --index {WORK}/eval_index.npz --report results/contamination.json --write-clean --labels \
    {DATA}/olmocr_docs/labels.jsonl {DATA}/olmocr_hist/labels.jsonl {DATA}/synthmix/labels.jsonl \
    {DATA}/render_old/labels.jsonl {DATA}/render_layout/labels.jsonl
!sed "s#data/#{DATA}/#g" mixtures/general-free-v1.json > {WORK}/mixture.json
!python scripts/make_data_card.py --mixture {WORK}/mixture.json --contamination results/contamination.json --out results/DATA_CARD.md
!head -30 results/DATA_CARD.md

## Phase 2c. General SFT (continues the Phase 1 adapter)

Measure first: `MAX_STEPS = 20` shows seconds per step; then set `-1` and choose `SAMPLES` (12,000 samples is roughly
4-6 hours on two T4s; check with the measured speed). Re-run the cell after a session ends: `--resume` continues.

In [ ]:
MAX_STEPS, SAMPLES = 20, 12000
launcher = f"accelerate launch --num_processes {N_GPUS} --mixed_precision fp16" if N_GPUS > 1 else "python"
!{launcher} scripts/train_sft.py --profile t4 --mixture {WORK}/mixture.json --init-adapter {PHASE1_ADAPTER} \
    --dev {DATA}/dev_old/labels.jsonl --dev-tests {DATA}/dev_old/tests.jsonl --eval-pages 12 \
    --out {RUNS}/general-t4 --samples-per-epoch {SAMPLES} --max-steps {MAX_STEPS} --save-every 150 --resume

## Gate G2

Dev sets: old scans (rendered), two-column pages (rendered) and real historical pages (`dev_loc`, CER only),
then the G0 olmOCR-Bench sample. The plan's rule: match or beat Flash on olmOCR-Bench, beat it on old scans and multi-column.

In [ ]:
ADAPTER = f"{RUNS}/general-t4/best" if os.path.exists(f"{RUNS}/general-t4/best") else f"{RUNS}/general-t4/adapter"
for d, extra in [("dev_old", ""), ("dev_multicol", "--where columns=2"), ("dev_loc", "")]:
    tests = f"--tests {DATA}/{d}/tests.jsonl" if os.path.exists(f"{DATA}/{d}/tests.jsonl") else ""
    !python scripts/bench_dev.py --labels {DATA}/{d}/labels.jsonl {tests} {extra} --name flash_{d} --limit 100 --batch-size 4 --max-new-tokens 4096
    !python scripts/bench_dev.py --labels {DATA}/{d}/labels.jsonl {tests} {extra} --name general_{d} --limit 100 --batch-size 4 --max-new-tokens 4096 --adapter {ADAPTER}
    !python scripts/compare.py results/dev/general_{d}.json results/dev/flash_{d}.json

In [ ]:
B = f"{WORK}/olmOCR-bench/bench_data"
if not os.path.exists("results/olmocr/flash_s300.json"):
    !python scripts/bench_olmocr.py --bench {B} --name flash_s300 --sample 300 --batch-size 4 --max-new-tokens 6000 --cache {WORK}/bench_cache
!python scripts/bench_olmocr.py --bench {B} --name general_s300 --sample 300 --batch-size 4 --max-new-tokens 6000 --cache {WORK}/bench_cache --adapter {ADAPTER}
!python scripts/compare.py results/olmocr/general_s300.json results/olmocr/flash_s300.json

## Phase 3. RL on page bands

Items are bands of rendered pages with their own unit tests (`scripts/make_rl_set.py`). Run the pilot, **read the dumped samples**
(`runs/rl-pilot/samples/`) for reward hacking, then the real run. Two T4s: each GPU samples its own items.

In [ ]:
!python scripts/make_rl_set.py --labels {DATA}/resumes/labels.jsonl {DATA}/render_old/labels.jsonl {DATA}/render_layout/labels.jsonl \
    --out {DATA}/rl_regions --per-page 2 --max-chars 1200 --limit 6000
L = f"torchrun --nproc_per_node {N_GPUS}" if N_GPUS > 1 else "python"
!{L} scripts/train_rl.py --profile t4 --adapter {ADAPTER} --data {DATA}/rl_regions/labels.jsonl --out {RUNS}/rl-pilot \
    --steps 50 --eval-every 25 --dump-every 25 --resume
!tail -3 {RUNS}/rl-pilot/train_log.jsonl
!head -c 3000 $(ls {RUNS}/rl-pilot/samples/*.jsonl | tail -1)

In [ ]:
RL_STEPS = 400
!{L} scripts/train_rl.py --profile t4 --adapter {ADAPTER} --data {DATA}/rl_regions/labels.jsonl --out {RUNS}/rl-t4 \
    --steps {RL_STEPS} --eval-every 50 --dump-every 100 --resume
!cat {RUNS}/rl-t4/dev_log.jsonl

## Gate G3

Tier B is decided against **our own** runs of Chandra 2 and Surya OCR 2 on the same sample (made in the Phase 0 notebook,
in a separate session because they pin their own libraries). Tier A must still hold on Resume-Bench.

In [ ]:
RL = f"{RUNS}/rl-t4/best"
!python scripts/bench_olmocr.py --bench {B} --name rl_s300 --sample 300 --batch-size 4 --max-new-tokens 6000 --cache {WORK}/bench_cache --adapter {RL}
for other in ["flash_s300", "chandra2_s300", "surya2_s300"]:
    if os.path.exists(f"results/olmocr/{other}.json"):
        !python scripts/compare.py results/olmocr/rl_s300.json results/olmocr/{other}.json

## Phase 4. Merge, quantise, measure

Merging runs on the CPU (~5 GB RAM). The 4-bit copy is what a consumer GPU runs, so its scores are re-measured.

In [ ]:
!python scripts/merge_lora.py --adapter {RL} --out {WORK}/models/folio-v1
!python scripts/merge_lora.py quantize --model {WORK}/models/folio-v1 --out {WORK}/models/folio-v1-nf4 --bits 4
!python scripts/bench_olmocr.py --bench {B} --name v1_nf4_s300 --sample 300 --batch-size 4 --max-new-tokens 6000 --cache {WORK}/bench_cache --model {WORK}/models/folio-v1-nf4
!python scripts/speed_test.py --images {WORK}/bench_cache/long1288 --pages 20 --name v1_t4 --model {WORK}/models/folio-v1 --batch-size 4
!python scripts/speed_test.py --images {WORK}/bench_cache/long1288 --pages 20 --name v1_nf4_t4 --model {WORK}/models/folio-v1-nf4 --batch-size 4

In [ ]:
!python scripts/make_model_card.py --name folio-ocr-v1 --model-dir {WORK}/models/folio-v1 --sft-run {RUNS}/general-t4 --rl-run {RUNS}/rl-t4 \
    --olmocr results/olmocr/rl_s300.json --olmocr-baseline $(ls results/olmocr/flash_s300.json results/olmocr/chandra2_s300.json results/olmocr/surya2_s300.json 2>/dev/null) \
    --speed results/speed/v1_t4.json results/speed/v1_nf4_t4.json --contamination results/contamination.json
!head -60 {WORK}/models/folio-v1/README.md

## Keep the results

Everything under `/kaggle/working` is saved with the notebook version (*Save Version*). The merged model is ~4-5 GB;
to share it: `hf upload <you>/folio-ocr-v1 /kaggle/working/models/folio-v1`. The holdout (gate G4) is built from fresh pages
with hand-checked tests: `scripts/draft_holdout_tests.py`, see `docs/PHASES.md`.